# Vector Databases and Retrieval

Notebook for **Chapter 6** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 A Tiny Retriever You Can Run (Exact Search and Metadata Filters)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
docs=[("d1","Refund requests are processed within 7 business days.","billing"),
("d2","To reset your password click Forgot Password on the login page.","account"),
("d3","Invoices are emailed on the first day of every month.","billing"),
("d4","Two factor authentication can be enabled in security settings.","account"),
("d5","Premium plan includes 24x7 phone support.","plans"),
("d6","Cancel your subscription any time from the billing page.","billing")]
tf=TfidfVectorizer(stop_words="english",sublinear_tf=True)
M=tf.fit_transform([d[1] for d in docs]).toarray()
M/=np.linalg.norm(M,axis=1,keepdims=True)
def retrieve(q,k=3,meta=None):
    v=tf.transform([q]).toarray()[0]; n=np.linalg.norm(v)
    v=v/n if n else v
    s=M@v
    order=[i for i in np.argsort(-s) if meta is None or docs[i][2]==meta]
    return [(docs[i][0],round(float(s[i]),3)) for i in order[:k]]
print(retrieve("how long do refunds take"))
print(retrieve("how do I reset my password"))
print(retrieve("billing page", meta="billing"))
print(retrieve("billing page", meta="plans"))

[('d1', 0.0), ('d2', 0.0), ('d3', 0.0)]
[('d2', 0.694), ('d1', 0.0), ('d3', 0.0)]
[('d6', 0.598), ('d1', 0.0), ('d3', 0.0)]
[('d5', 0.0)]


In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
docs=[("d1","Refund requests are processed within 7 business days.","billing"),
("d2","To reset your password click Forgot Password on the login page.","account"),
("d3","Invoices are emailed on the first day of every month.","billing"),
("d4","Two factor authentication can be enabled in security settings.","account"),
("d5","Premium plan includes 24x7 phone support.","plans"),
("d6","Cancel your subscription any time from the billing page.","billing")]
def tok(text):
    words=re.findall(r"[a-z0-9]+",text.lower())
    return [w[:-1] if len(w)>3 and w.endswith("s") else w for w in words]
tf=TfidfVectorizer(tokenizer=tok,token_pattern=None,stop_words=None,sublinear_tf=True)
M=tf.fit_transform([d[1] for d in docs]).toarray()
M/=np.linalg.norm(M,axis=1,keepdims=True)
def retrieve(q,k=3,meta=None):
    v=tf.transform([q]).toarray()[0]; n=np.linalg.norm(v)
    v=v/n if n else v
    s=M@v
    order=[i for i in np.argsort(-s) if meta is None or docs[i][2]==meta]
    return [(docs[i][0],round(float(s[i]),3)) for i in order[:k]]
print(retrieve("how long do refunds take"))
print(retrieve("how do I reset my password"))
print(retrieve("billing page", meta="billing"))
print(retrieve("billing page", meta="plans"))

[('d1', 0.369), ('d2', 0.0), ('d3', 0.0)]
[('d2', 0.592), ('d1', 0.0), ('d3', 0.0)]
[('d6', 0.462), ('d1', 0.0), ('d3', 0.0)]
[('d5', 0.0)]


In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
docs=["Refund requests are processed within 7 business days.",
"To reset your password click Forgot Password on the login page.",
"Invoices are emailed on the first day of every month.",
"Two factor authentication can be enabled in security settings.",
"Premium plan includes 24x7 phone support.",
"Cancel your subscription any time from the billing page."]
def tok(t):
    return [w[:-1] if len(w)>3 and w.endswith("s") else w for w in re.findall(r"[a-z0-9]+",t.lower())]
tf=TfidfVectorizer(tokenizer=tok,token_pattern=None,sublinear_tf=True)
M=tf.fit_transform(docs).toarray(); M/=np.linalg.norm(M,axis=1,keepdims=True)
queries=[("how long do refunds take",True),("forgot my password",True),("when are invoices sent",True),
         ("does the premium plan have phone support",True),
         ("what is the weather in Pune",False),("can I pay with bitcoin",False),
         ("who is the CEO of the company",False),("how do I export my data",False)]
rows=[]
for q,ans in queries:
    v=tf.transform([q]).toarray()[0]; n=np.linalg.norm(v); v=v/n if n else v
    s=M@v; rows.append((q,ans,float(s.max()),int(s.argmax())))
    print(f"{s.max():.3f}  top=d{s.argmax()+1}  answerable={ans!s:5}  {q}")
for th in [0.0,0.1,0.2,0.3,0.4]:
    answered=[r for r in rows if r[2]>th]
    good=sum(1 for r in answered if r[1]); bad=len(answered)-good
    missed=sum(1 for r in rows if r[1] and r[2]<=th)
    print(f"threshold {th:.1f}: answered {len(answered)}, of which unanswerable (hallucination risk)={bad}, answerable questions refused={missed}")

0.369  top=d1  answerable=True   how long do refunds take
0.592  top=d2  answerable=True   forgot my password
0.444  top=d3  answerable=True   when are invoices sent
0.772  top=d5  answerable=True   does the premium plan have phone support
0.274  top=d4  answerable=False  what is the weather in Pune
0.333  top=d4  answerable=False  can I pay with bitcoin
0.403  top=d3  answerable=False  who is the CEO of the company
0.000  top=d1  answerable=False  how do I export my data
threshold 0.0: answered 7, of which unanswerable (hallucination risk)=3, answerable questions refused=0
threshold 0.1: answered 7, of which unanswerable (hallucination risk)=3, answerable questions refused=0
threshold 0.2: answered 7, of which unanswerable (hallucination risk)=3, answerable questions refused=0
threshold 0.3: answered 6, of which unanswerable (hallucination risk)=2, answerable questions refused=0
threshold 0.4: answered 4, of which unanswerable (hallucination risk)=1, answerable questions refused=1


## 6.2 Approximate Nearest Neighbour Indexes: Flat, IVF, HNSW, PQ

In [ ]:
import numpy as np, time
rng=np.random.default_rng(0)
N,d=50000,64
centers=rng.normal(size=(200,d))*3
X=(centers[rng.integers(0,200,N)]+rng.normal(size=(N,d))).astype("float32")
X/=np.linalg.norm(X,axis=1,keepdims=True)
Q=X[rng.choice(N,100,replace=False)]+0.1*rng.normal(size=(100,d)).astype("float32")
Q/=np.linalg.norm(Q,axis=1,keepdims=True)
t=time.perf_counter(); S=Q@X.T; truth=np.argsort(-S,axis=1)[:,:10]; tf=time.perf_counter()-t
print(f"flat exact: {tf*1000/100:.2f} ms/query (batched)")
K=100
C=X[rng.choice(N,K,replace=False)].copy()
for _ in range(8):                                   # a few k-means rounds
    a=np.argmax(X@C.T,axis=1)
    for j in range(K):
        m=X[a==j]
        if len(m): c=m.mean(0); C[j]=c/np.linalg.norm(c)
a=np.argmax(X@C.T,axis=1)
lists=[np.nonzero(a==j)[0] for j in range(K)]
for nprobe in [1,3,10]:
    t=time.perf_counter(); hit=0; scanned=0
    for qi,q in enumerate(Q):
        cs=np.argsort(-(C@q))[:nprobe]
        cand=np.concatenate([lists[j] for j in cs]); scanned+=len(cand)
        top=cand[np.argsort(-(X[cand]@q))[:10]]
        hit+=len(set(top)&set(truth[qi]))
    el=time.perf_counter()-t
    print(f"nprobe={nprobe:2d} recall@10={hit/1000:.3f} scanned={scanned/100/N:.1%} of data, {el*10:.2f} ms/query")

flat exact: 3.17 ms/query (batched)
nprobe= 1 recall@10=0.970 scanned=1.2% of data, 0.14 ms/query
nprobe= 3 recall@10=0.990 scanned=3.3% of data, 0.21 ms/query
nprobe=10 recall@10=1.000 scanned=10.3% of data, 0.58 ms/query


## 6.3 Hybrid Search and Reranking

In [ ]:
def rrf(rankings,k=60):
    s={}
    for r in rankings:
        for pos,d in enumerate(r,1): s[d]=s.get(d,0)+1/(k+pos)
    return sorted(s.items(),key=lambda x:-x[1])
bm25=["d3","d1","d7","d2"]; vec=["d1","d5","d3","d9"]
for d,sc in rrf([bm25,vec]): print(d,round(sc,5))

d1 0.03252
d3 0.03227
d5 0.01613
d7 0.01587
d2 0.01562
d9 0.01562


## 6.4 From Zero: What a Vector Search Really Computes (Cosine, Dot Product, Distance and the Cost of Exact Search)

In [ ]:
import numpy as np
a=np.array([3.0,4.0]); b=np.array([4.0,3.0]); c=np.array([6.0,8.0])
def cos(x,y): return x@y/(np.linalg.norm(x)*np.linalg.norm(y))
for name,y in [("b",b),("c",c)]:
    print(f"a vs {name}: dot={a@y:.0f}  cosine={cos(a,y):.3f}  L2={np.linalg.norm(a-y):.3f}")
# after scaling every vector to length 1
u=lambda v: v/np.linalg.norm(v)
ua,ub,uc=u(a),u(b),u(c)
print("unit a.b =",round(ua@ub,3)," L2^2 =",round(np.sum((ua-ub)**2),3)," 2-2*cos =",round(2-2*(ua@ub),3))
print("unit a.c =",round(ua@uc,3)," L2^2 =",round(np.sum((ua-uc)**2),3))

a vs b: dot=24  cosine=0.960  L2=1.414
a vs c: dot=50  cosine=1.000  L2=5.000
unit a.b = 0.96  L2^2 = 0.08  2-2*cos = 0.08
unit a.c = 1.0  L2^2 = 0.0


In [ ]:
import numpy as np
rng=np.random.default_rng(1)
X=rng.normal(size=(1000,16))*rng.uniform(0.5,3,size=(1000,1))   # vectors of very different lengths
q=rng.normal(size=16)
def top5(scores): return np.argsort(-scores)[:5].tolist()
cos=(X@q)/(np.linalg.norm(X,axis=1)*np.linalg.norm(q))
dot=X@q
l2=-np.linalg.norm(X-q,axis=1)
print("raw vectors")
print(" cosine top5:",top5(cos)); print(" dot    top5:",top5(dot)); print(" -L2    top5:",top5(l2))
U=X/np.linalg.norm(X,axis=1,keepdims=True); uq=q/np.linalg.norm(q)
print("after normalising")
print(" cosine top5:",top5(U@uq)); print(" dot    top5:",top5(U@uq)); print(" -L2    top5:",top5(-np.linalg.norm(U-uq,axis=1)))

raw vectors
 cosine top5: [180, 120, 345, 483, 186]
 dot    top5: [483, 53, 689, 487, 649]
 -L2    top5: [345, 685, 292, 94, 853]
after normalising
 cosine top5: [180, 120, 345, 483, 186]
 dot    top5: [180, 120, 345, 483, 186]
 -L2    top5: [180, 120, 345, 483, 186]


In [ ]:
# cost of exact search for ONE query: N*d multiply-adds, and N*d*4 bytes that must be read from memory
rows=[(10_000,384),(1_000_000,768),(50_000_000,768),(1_000_000_000,1024)]
print(f"{'N':>14} {'dim':>5} {'GB float32':>11} {'GFLOP':>9} {'s @50 GB/s':>11}")
for N,d in rows:
    gb=N*d*4/1e9; gflop=2*N*d/1e9
    print(f"{N:>14,} {d:>5} {gb:>11.2f} {gflop:>9.2f} {gb/50:>11.3f}")

             N   dim  GB float32     GFLOP  s @50 GB/s
        10,000   384        0.02      0.01       0.000
     1,000,000   768        3.07      1.54       0.061
    50,000,000   768      153.60     76.80       3.072
 1,000,000,000  1024     4096.00   2048.00      81.920


In [ ]:
import numpy as np
rng=np.random.default_rng(0)
print("dim  nearest  farthest  farthest/nearest")
for d in [2,10,100,1000]:
    X=rng.normal(size=(5000,d)); q=rng.normal(size=d)
    dist=np.linalg.norm(X-q,axis=1)
    print(f"{d:>4} {dist.min():8.2f} {dist.max():9.2f} {dist.max()/dist.min():12.2f}")

dim  nearest  farthest  farthest/nearest
   2     0.03      4.90       146.87
  10     1.46      7.39         5.05
 100    10.25     16.22         1.58
1000    42.18     48.31         1.15


## 6.5 Inside the Indexes: IVF, HNSW and PQ Built and Measured

In [ ]:
import os
os.environ["OMP_NUM_THREADS"]="1"          # one thread, set before importing numpy: k-means then gives the same answer on every run
import numpy as np
from sklearn.cluster import KMeans
def make(spread, N=30000, d=32, groups=60, seed=0):
    rng=np.random.default_rng(seed)
    centers=rng.normal(size=(groups,d))*spread
    X=centers[rng.integers(0,groups,N)]+rng.normal(size=(N,d))
    X=(X/np.linalg.norm(X,axis=1,keepdims=True)).astype("float32")
    Q=X[rng.choice(N,200,replace=False)]+0.15*rng.normal(size=(200,d)).astype("float32")
    return X,(Q/np.linalg.norm(Q,axis=1,keepdims=True)).astype("float32")
def run(name,spread,nlist=100,k=10):
    X,Q=make(spread)
    truth=np.argsort(-(Q@X.T),axis=1)[:,:k]                  # exact answers
    km=KMeans(nlist,n_init=1,random_state=0).fit(X)           # build the lists
    C=km.cluster_centers_.astype("float32"); assign=km.labels_
    lists=[np.nonzero(assign==j)[0] for j in range(nlist)]
    print(name)
    for nprobe in [1,2,5,10,20,40]:
        hit=0; scanned=0
        for qi,q in enumerate(Q):
            near=np.argsort(-(C@q))[:nprobe]
            cand=np.concatenate([lists[j] for j in near]); scanned+=len(cand)
            top=cand[np.argsort(-(X[cand]@q))[:k]]
            hit+=len(set(top.tolist())&set(truth[qi].tolist()))
        print(f"  nprobe={nprobe:>2}  recall@10={hit/(len(Q)*k):.3f}  scanned={scanned/len(Q)/len(X):.1%}")
run("well separated groups",3.0)
run("overlapping groups (more like real text)",0.8)

well separated groups
  nprobe= 1  recall@10=0.812  scanned=1.2%
  nprobe= 2  recall@10=0.975  scanned=2.1%
  nprobe= 5  recall@10=1.000  scanned=5.2%
  nprobe=10  recall@10=1.000  scanned=10.1%
  nprobe=20  recall@10=1.000  scanned=20.0%
  nprobe=40  recall@10=1.000  scanned=39.7%
overlapping groups (more like real text)
  nprobe= 1  recall@10=0.502  scanned=1.1%
  nprobe= 2  recall@10=0.680  scanned=2.1%
  nprobe= 5  recall@10=0.835  scanned=5.2%
  nprobe=10  recall@10=0.916  scanned=10.1%
  nprobe=20  recall@10=0.972  scanned=20.2%
  nprobe=40  recall@10=0.997  scanned=40.2%


In [ ]:
import numpy as np
P={"A":(0,0),"B":(2,1),"C":(4,1),"D":(6,2),"E":(7,5),"F":(5,6),"G":(8,3)}
edges=[("A","B"),("B","C"),("C","D"),("D","E"),("D","G"),("C","F"),("F","E"),("E","G")]
nb={k:set() for k in P}
for u,v in edges: nb[u].add(v); nb[v].add(u)
def dist(a,q): return float(np.hypot(P[a][0]-q[0],P[a][1]-q[1]))
def greedy(q,start,graph):
    cur=start; calls=1; path=[cur]; print(f"  start {cur}: dist={dist(cur,q):.3f}")
    while True:
        best,bd=cur,dist(cur,q)
        for v in sorted(graph[cur]):
            calls+=1; dv=dist(v,q); print(f"    at {cur}: look at {v}, dist={dv:.3f}")
            if dv<bd: best,bd=v,dv
        if best==cur: return cur,path,calls
        cur=best; path.append(cur); print(f"  move to {cur}")
q=(7,4)
print("query (7,4), full graph")
end,path,calls=greedy(q,"A",nb)
print("stopped at",end,"path",path,"distance computations",calls)
true=min(P,key=lambda a:dist(a,q)); print("true nearest:",true)
# remove the edge D-E and E-G, so the way to the goal is blocked by a "valley"
nb2={k:set(v) for k,v in nb.items()}
nb2["D"].discard("E"); nb2["E"].discard("D"); nb2["E"].discard("G"); nb2["G"].discard("E")
print("same query, graph without edges D-E and E-G")
end,path,calls=greedy(q,"A",nb2)
print("stopped at",end,"path",path,"-> true nearest is",true)
# beam search with ef=3 keeps several candidates alive, so it can leave the dead end
import heapq
def beam(q,start,graph,ef):
    seen={start}; cand=[(dist(start,q),start)]; best=[(dist(start,q),start)]
    while cand:
        d,u=heapq.heappop(cand)
        if len(best)>=ef and d>max(best)[0]: break
        for v in sorted(graph[u]):
            if v in seen: continue
            seen.add(v); dv=dist(v,q)
            if len(best)<ef or dv<max(best)[0]:
                heapq.heappush(cand,(dv,v)); best.append((dv,v)); best=sorted(best)[:ef]
    return [(v,round(d,3)) for d,v in sorted(best)]
for ef in (1,2,3):
    print(f"beam search ef={ef} on the broken graph:",beam(q,"A",nb2,ef))

query (7,4), full graph
  start A: dist=8.062
    at A: look at B, dist=5.831
  move to B
    at B: look at A, dist=8.062
    at B: look at C, dist=4.243
  move to C
    at C: look at B, dist=5.831
    at C: look at D, dist=2.236
    at C: look at F, dist=2.828
  move to D
    at D: look at C, dist=4.243
    at D: look at E, dist=1.000
    at D: look at G, dist=1.414
  move to E
    at E: look at D, dist=2.236
    at E: look at F, dist=2.828
    at E: look at G, dist=1.414
stopped at E path ['A', 'B', 'C', 'D', 'E'] distance computations 13
true nearest: E
same query, graph without edges D-E and E-G
  start A: dist=8.062
    at A: look at B, dist=5.831
  move to B
    at B: look at A, dist=8.062
    at B: look at C, dist=4.243
  move to C
    at C: look at B, dist=5.831
    at C: look at D, dist=2.236
    at C: look at F, dist=2.828
  move to D
    at D: look at C, dist=4.243
    at D: look at G, dist=1.414
  move to G
    at G: look at D, dist=2.236
stopped at G path ['A', 'B', 'C', '

In [ ]:
import numpy as np, heapq
rng=np.random.default_rng(0)
N,d,M=20000,32,12
centers=rng.normal(size=(60,d))*0.8
X=centers[rng.integers(0,60,N)]+rng.normal(size=(N,d))
X=(X/np.linalg.norm(X,axis=1,keepdims=True)).astype("float32")
Q=X[rng.choice(N,100,replace=False)]+0.15*rng.normal(size=(100,d)).astype("float32")
Q/=np.linalg.norm(Q,axis=1,keepdims=True)
truth=np.argsort(-(Q@X.T),axis=1)[:,:10]

# ---- build: each node links to its M nearest neighbours (exact kNN graph, made two-way)
def knn_graph(ids,M):
    P=X[ids]; S=P@P.T; np.fill_diagonal(S,-9)
    nb=np.argsort(-S,axis=1)[:,:M]
    g={int(ids[i]):set(int(ids[j]) for j in nb[i]) for i in range(len(ids))}
    for u in list(g):
        for v in g[u]: g[v].add(u)
    return g
top_ids=rng.choice(N,N//16,replace=False)             # about 1/16 of nodes also live in an upper layer
upper=knn_graph(top_ids,M)
bottom={i:set() for i in range(N)}
for s in range(0,N,1000):                              # build in blocks of 1000 rows to save memory
    blk=X[s:s+1000]@X.T
    for r in range(blk.shape[0]): blk[r,s+r]=-9
    nb=np.argpartition(-blk,M,axis=1)[:,:M]
    for r in range(blk.shape[0]):
        for j in nb[r]:
            bottom[s+r].add(int(j)); bottom[int(j)].add(s+r)

# ---- search
def greedy(graph,q,start):
    cur=start; best=float(X[cur]@q); dist_calls=1
    while True:
        improved=False
        for v in graph[cur]:
            dist_calls+=1; sc=float(X[v]@q)
            if sc>best: best,cur,improved=sc,v,True
        if not improved: return cur,dist_calls
def beam(graph,q,start,ef,allow=None):
    seen={start}; calls=1; sc0=float(X[start]@q)
    cand=[(-sc0,start)]                               # cand: max-heap by score
    res=[(sc0,start)] if allow is None or allow[start] else []   # res: min-heap of best ef allowed nodes
    while cand:
        negs,u=heapq.heappop(cand)
        if len(res)>=ef and -negs<res[0][0]: break
        for v in graph[u]:
            if v in seen: continue
            seen.add(v); calls+=1; sc=float(X[v]@q)
            if len(res)<ef or sc>res[0][0]:
                heapq.heappush(cand,(-sc,v))
                if allow is None or allow[v]:
                    heapq.heappush(res,(sc,v))
                    if len(res)>ef: heapq.heappop(res)
    return [v for _,v in sorted(res,reverse=True)],calls
def search(q,ef,k=10,allow=None):
    entry,c1=greedy(upper,q,int(top_ids[0]))
    out,c2=beam(bottom,q,entry,ef,allow)
    return out[:k],c1+c2

print(f"graph: N={N}, M={M}, average links per node={np.mean([len(v) for v in bottom.values()]):.1f}")
for ef in [10,20,40,80,160]:
    hit=0; calls=0
    for qi,q in enumerate(Q):
        r,c=search(q,ef); calls+=c; hit+=len(set(r)&set(truth[qi].tolist()))
    print(f"efSearch={ef:>3}  recall@10={hit/1000:.3f}  distance computations/query={calls/100:>7.0f} ({calls/100/N:.1%} of N)")

# ---- filters: (a) search first, drop non-matching afterwards; (b) check the filter while walking the graph
print("filter  efSearch | post-filter: results/10  recall | filter-aware: results/10  recall  dist.calcs")
for sel in [0.5,0.1,0.01]:
    allow=rng.random(N)<sel                            # random attribute, unrelated to the vectors
    ok=np.nonzero(allow)[0]
    for ef in [40,200]:
        n_a=n_b=h_a=h_b=tot=calls=0
        for q in Q[:50]:
            exact=set(ok[np.argsort(-(X[ok]@q))[:10]].tolist())
            ra,_=search(q,ef,k=ef)                     # unfiltered walk, keep ef candidates
            ra=[v for v in ra if allow[v]][:10]
            rb,cb=search(q,ef,allow=allow); calls+=cb
            n_a+=len(ra); n_b+=len(rb); h_a+=len(set(ra)&exact); h_b+=len(set(rb)&exact); tot+=len(exact)
        print(f"{sel:>5.0%}   {ef:>6}   | {n_a/50:>14.1f} {h_a/tot:>10.3f} | {n_b/50:>17.1f} {h_b/tot:>8.3f} {calls/50:>10.0f}")

graph: N=20000, M=12, average links per node=17.5
efSearch= 10  recall@10=0.644  distance computations/query=    273 (1.4% of N)
efSearch= 20  recall@10=0.769  distance computations/query=    388 (1.9% of N)
efSearch= 40  recall@10=0.862  distance computations/query=    571 (2.9% of N)
efSearch= 80  recall@10=0.934  distance computations/query=    883 (4.4% of N)
efSearch=160  recall@10=0.979  distance computations/query=   1447 (7.2% of N)
filter  efSearch | post-filter: results/10  recall | filter-aware: results/10  recall  dist.calcs
  50%       40   |           10.0      0.834 |              10.0    0.914        889
  50%      200   |           10.0      0.980 |              10.0    0.998       3067
  10%       40   |            4.0      0.396 |              10.0    0.984       3149
  10%      200   |           10.0      0.948 |              10.0    1.000      10086
   1%       40   |            0.4      0.036 |              10.0    1.000      13899
   1%      200   |            2.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"]="1"          # one thread, set before importing numpy: k-means then gives the same answer on every run
import numpy as np
from sklearn.cluster import KMeans
rng=np.random.default_rng(0)
N,d,groups=20000,64,60
centers=rng.normal(size=(groups,d))*0.8
X=centers[rng.integers(0,groups,N)]+rng.normal(size=(N,d))
X=(X/np.linalg.norm(X,axis=1,keepdims=True)).astype("float32")
Q=X[rng.choice(N,100,replace=False)]+0.15*rng.normal(size=(100,d)).astype("float32")
Q/=np.linalg.norm(Q,axis=1,keepdims=True)
truth=np.argsort(-(Q@X.T),axis=1)[:,:10]

def train_pq(X,m,ks=256):
    sub=X.shape[1]//m; books=[]; codes=np.zeros((len(X),m),dtype=np.uint8)
    for j in range(m):
        part=X[:,j*sub:(j+1)*sub]
        km=KMeans(ks,n_init=1,max_iter=25,random_state=0).fit(part)
        books.append(km.cluster_centers_.astype("float32")); codes[:,j]=km.labels_
    return books,codes
def adc_scores(q,books,codes):
    sub=len(q)//len(books)
    # lookup table: dot product of each query piece with all 256 centroids of that piece
    table=np.stack([books[j]@q[j*sub:(j+1)*sub] for j in range(len(books))])   # shape (m, 256)
    return table[np.arange(len(books)),codes].sum(axis=1)                      # add m table entries per vector

print(f"full vectors: {d*4} bytes each = {N*d*4/1e6:.1f} MB")
for m in [4,8,16]:
    books,codes=train_pq(X,m)
    approx=np.hstack([books[j][codes[:,j]] for j in range(m)])
    err=np.mean(np.linalg.norm(X-approx,axis=1))
    h1=h2=0
    for qi,q in enumerate(Q):
        s=adc_scores(q,books,codes)
        top=np.argsort(-s)
        h1+=len(set(top[:10].tolist())&set(truth[qi].tolist()))
        cand=top[:100]                                   # rescore 100 candidates with the full vectors
        best=cand[np.argsort(-(X[cand]@q))[:10]]
        h2+=len(set(best.tolist())&set(truth[qi].tolist()))
    print(f"m={m:>2}: {m} bytes/vector ({d*4//m}x smaller), index {N*m/1e6:.2f} MB, "
          f"mean reconstruction error {err:.3f}, recall@10 PQ only={h1/1000:.3f}, PQ + rescore 100={h2/1000:.3f}")

full vectors: 256 bytes each = 5.1 MB
m= 4: 4 bytes/vector (64x smaller), index 0.08 MB, mean reconstruction error 0.686, recall@10 PQ only=0.181, PQ + rescore 100=0.597
m= 8: 8 bytes/vector (32x smaller), index 0.16 MB, mean reconstruction error 0.529, recall@10 PQ only=0.352, PQ + rescore 100=0.833
m=16: 16 bytes/vector (16x smaller), index 0.32 MB, mean reconstruction error 0.293, recall@10 PQ only=0.596, PQ + rescore 100=0.995


## 6.6 Filters, Permissions and Score Thresholds

In [ ]:
import sqlite3, numpy as np
rng=np.random.default_rng(3)
N,d=3000,32
E=rng.normal(size=(N,d)).astype("float32"); E/=np.linalg.norm(E,axis=1,keepdims=True)
depts=rng.choice(["billing","account","plans","legal"],N,p=[0.45,0.3,0.2,0.05])
years=rng.choice([2023,2024,2025],N,p=[0.2,0.3,0.5])
con=sqlite3.connect(":memory:")
con.execute("CREATE TABLE chunks(id INTEGER PRIMARY KEY, dept TEXT, year INT, emb BLOB)")
con.executemany("INSERT INTO chunks VALUES(?,?,?,?)",
                [(i,str(depts[i]),int(years[i]),E[i].tobytes()) for i in range(N)])
con.execute("CREATE INDEX idx_dept_year ON chunks(dept,year)")

def search_pre(q,dept,year,k=5):
    """Pre-filter: SQL picks the allowed rows first, then we score only those."""
    rows=con.execute("SELECT id, emb FROM chunks WHERE dept=? AND year=?",(dept,year)).fetchall()
    if not rows: return [],0
    ids=np.array([r[0] for r in rows]); M=np.vstack([np.frombuffer(r[1],dtype="float32") for r in rows])
    s=M@q; o=np.argsort(-s)[:k]
    return [(int(ids[i]),round(float(s[i]),3)) for i in o],len(rows)

def search_post(q,dept,year,k=5):
    """Post-filter: take the global top k, then drop rows that fail the filter."""
    top=np.argsort(-(E@q))[:k]
    keep=[int(i) for i in top if depts[i]==dept and years[i]==year]
    return keep

q=E[7]
print("query row 7 has dept=%s year=%d"%(depts[7],years[7]))
print("pre-filter  legal/2023:",search_pre(q,"legal",2023))
print("post-filter legal/2023:",search_post(q,"legal",2023))
print("pre-filter  billing/2025:",search_pre(q,"billing",2025))
print("post-filter billing/2025:",search_post(q,"billing",2025))
# how many rows does each filter keep?
for dept,year in [("billing",2025),("legal",2023)]:
    n=con.execute("SELECT COUNT(*) FROM chunks WHERE dept=? AND year=?",(dept,year)).fetchone()[0]
    print(f"{dept}/{year}: {n} of {N} rows = {n/N:.1%}")

query row 7 has dept=legal year=2025
pre-filter  legal/2023: ([(2131, 0.295), (2998, 0.256), (864, 0.229), (1586, 0.223), (1337, 0.102)], 25)
post-filter legal/2023: []
pre-filter  billing/2025: ([(186, 0.643), (2706, 0.538), (1756, 0.51), (576, 0.505), (2060, 0.449)], 707)
post-filter billing/2025: [186, 2706]
billing/2025: 707 of 3000 rows = 23.6%
legal/2023: 25 of 3000 rows = 0.8%


## 6.7 Hybrid Search in Depth: BM25, RRF and When Each Retriever Fails

In [ ]:
import math
# BM25 for the query "refund" against three chunks, by hand then in code. k1=1.5, b=0.75
docs={"A":"refund requests are processed within 7 business days".split(),
      "C":"two factor authentication can be enabled in security settings".split(),
      "B":"invoices are emailed on the first day of every month and refund rules are on the billing page".split()}
n=len(docs); avg=sum(len(t) for t in docs.values())/n
df=sum("refund" in t for t in docs.values())
idf=math.log(1+(n-df+0.5)/(df+0.5))
print("N =",n," avg length =",round(avg,3)," df(refund) =",df," idf =",round(idf,4))
for name,t in sorted(docs.items()):
    tf=t.count("refund"); L=len(t)
    norm=1.5*(1-0.75+0.75*L/avg)
    s=idf*tf*(1.5+1)/(tf+norm)
    print(f"chunk {name}: length={L}  tf={tf}  length-norm term={norm:.4f}  BM25={s:.4f}")

N = 3  avg length = 11.667  df(refund) = 2  idf = 0.47
chunk A: length=8  tf=1  length-norm term=1.1464  BM25=0.5474
chunk B: length=18  tf=1  length-norm term=2.1107  BM25=0.3777
chunk C: length=9  tf=0  length-norm term=1.2429  BM25=0.0000


In [ ]:
import re, math
import numpy as np
from collections import Counter
docs={"h1":"Refund requests are processed within 7 business days.",
"h2":"Money is returned to the original payment method after approval.",
"h3":"Error XJ-4471 means the payment gateway timed out; retry after five minutes.",
"h4":"Error XJ-4472 means the card was declined by the issuing bank.",
"h5":"To reset your password click Forgot Password on the login page.",
"h6":"Two factor authentication can be enabled in security settings.",
"h7":"Premium plan includes 24x7 phone support.",
"h8":"Cancel your subscription any time from the billing page.",
"h9":"Invoices are emailed on the first day of every month.",
"h10":"You can close your account permanently from privacy settings."}
queries=[("when will I get reimbursed","h2"),("I lost my credentials","h5"),
("erase everything about me","h10"),("stop my plan from renewing","h8"),     # paraphrases
("XJ-4471","h3"),("XJ-4472","h4"),("what does XJ-4472 mean","h4"),          # exact codes
("how long do refunds take","h1"),("extra login security","h6"),("payment error XJ-4471","h3")]   # mixed
ids=list(docs)

def words(t): return [w[:-1] if len(w)>3 and w.endswith("s") else w
                      for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*",t.lower())]
# ---- BM25 from scratch
toks={i:words(t) for i,t in docs.items()}
avg=np.mean([len(t) for t in toks.values()]); n=len(docs)
df=Counter(w for t in toks.values() for w in set(t))
def bm25(q,k1=1.5,b=0.75):
    sc=[]
    for i in ids:
        tf=Counter(toks[i]); s=0.0
        for w in set(words(q)):
            if w in tf:
                idf=math.log(1+(n-df[w]+0.5)/(df[w]+0.5))
                s+=idf*tf[w]*(k1+1)/(tf[w]+k1*(1-b+b*len(toks[i])/avg))
        sc.append(s)
    return np.array(sc)
# ---- "dense" stand-in: synonyms share a concept token, digits are dropped (codes blur together)
CONCEPT={"refund":"REFUND","reimbursed":"REFUND","reimburse":"REFUND","money":"REFUND","return":"REFUND","returned":"REFUND","back":"REFUND",
 "password":"ACCESS","login":"ACCESS","credential":"ACCESS","reset":"ACCESS",
 "cancel":"CANCEL","stop":"CANCEL","renewing":"CANCEL","subscription":"CANCEL",
 "delete":"REMOVE","erase":"REMOVE","close":"REMOVE","permanently":"REMOVE",
 "security":"SECURE","authentication":"SECURE","factor":"SECURE"}
def dense_words(t):
    out=[]
    for w in words(t):
        w=re.sub(r"-?[0-9]+","",w)
        if w: out.append(CONCEPT.get(w,w))
    return out
from sklearn.feature_extraction.text import TfidfVectorizer
tv=TfidfVectorizer(tokenizer=dense_words,token_pattern=None,sublinear_tf=True)
D=tv.fit_transform(docs.values()).toarray(); D/=np.linalg.norm(D,axis=1,keepdims=True)
def dense(q):
    v=tv.transform([q]).toarray()[0]; nn=np.linalg.norm(v); return D@(v/nn if nn else v)
def order(scores):                      # only chunks with a positive score are "retrieved"
    return [ids[i] for i in np.argsort(-scores,kind="stable") if scores[i]>0]
def rank_of(scores,target):
    o=order(scores); return o.index(target)+1 if target in o else 99   # 99 = not retrieved
def rrf(rank_lists,k=60):
    s=Counter()
    for r in rank_lists:
        for pos,d in enumerate(r,1): s[d]+=1/(k+pos)
    return s
print(f"{'query':38} {'BM25':>5} {'dense':>6} {'hybrid':>7}   (rank of the right chunk, 99 = not retrieved)")
res={"bm25":[], "dense":[], "hybrid":[]}
for q,t in queries:
    rb=rank_of(bm25(q),t); rd=rank_of(dense(q),t)
    fused=rrf([order(bm25(q)),order(dense(q))]); fo=sorted(fused,key=lambda d:-fused[d]); rh=fo.index(t)+1 if t in fo else 99
    res["bm25"].append(rb); res["dense"].append(rd); res["hybrid"].append(rh)
    print(f"{q:38} {rb:>5} {rd:>6} {rh:>7}")
for m,r in res.items():
    r=np.where(np.array(r)==99,np.inf,np.array(r)); print(f"{m:7} recall@3={np.mean(r<=3):.3f}  MRR={np.mean(1/r):.3f}")

query                                   BM25  dense  hybrid   (rank of the right chunk, 99 = not retrieved)
when will I get reimbursed                99      1       1
I lost my credentials                     99      1       1
erase everything about me                 99      1       1
stop my plan from renewing                 2      1       2
XJ-4471                                    1      2       1
XJ-4472                                    1      1       1
what does XJ-4472 mean                     1      1       1
how long do refunds take                   1      2       1
extra login security                       1      2       1
payment error XJ-4471                      1      1       1
bm25    recall@3=0.700  MRR=0.650
dense   recall@3=1.000  MRR=0.850
hybrid  recall@3=1.000  MRR=0.950


## 6.8 Reranking, the Candidate Funnel and Compression Tricks

In [ ]:
import numpy as np
rng=np.random.default_rng(7)
Q,C=2000,1000                                   # 2000 queries, 1000 candidate chunks each, 1 relevant chunk per query
s1=rng.normal(size=(Q,C)); s1[:,0]+=3.0         # stage-1 score: relevant chunk (column 0) gets a +3.0 boost
s2=rng.normal(size=(Q,C)); s2[:,0]+=5.0         # reranker score: independent noise, stronger boost +5.0
order1=np.argsort(-s1,axis=1)                   # stage-1 ranking of each query
print(f"{'depth':>6} {'stage-1 recall@depth':>21} {'final recall@5':>15} {'pairs reranked':>15} {'seconds @20ms/pair':>19}")
for depth in [5,10,20,50,100,200,1000]:
    cand=order1[:,:depth]                                   # candidates handed to the reranker
    r1=np.mean((cand==0).any(axis=1))                       # was the relevant chunk among them?
    rr=np.take_along_axis(s2,cand,axis=1)                   # reranker scores of those candidates only
    best5=np.take_along_axis(cand,np.argsort(-rr,axis=1)[:,:5],axis=1)
    r2=np.mean((best5==0).any(axis=1))
    print(f"{depth:>6} {r1:>21.3f} {r2:>15.3f} {depth:>15} {depth*0.020:>19.1f}")

 depth  stage-1 recall@depth  final recall@5  pairs reranked  seconds @20ms/pair
     5                 0.652           0.652               5                 0.1
    10                 0.747           0.747              10                 0.2
    20                 0.837           0.837              20                 0.4
    50                 0.919           0.918              50                 1.0
   100                 0.960           0.959             100                 2.0
   200                 0.989           0.987             200                 4.0
  1000                 1.000           0.989            1000                20.0


In [ ]:
import numpy as np
# ColBERT-style late interaction by hand: one small vector per token, score = sum over query tokens of the best match
q=np.array([[1.0,0.0],[0.0,1.0]])                      # 2 query tokens (2-dim toy vectors, unit length)
docA=np.array([[0.8,0.6],[0.6,0.8],[1.0,0.0]])           # 3 document tokens
docB=np.array([[0.0,1.0],[0.6,0.8]])                     # 2 document tokens
for name,D in [("A",docA),("B",docB)]:
    sim=q@D.T                                          # (query tokens x doc tokens) cosine matrix
    best=sim.max(axis=1)                               # best doc token for every query token
    print(f"doc {name}: similarity matrix=\n{sim}\n  best per query token={best}  MaxSim score={best.sum():.2f}")
# a single-vector (bi-encoder) comparison: average the token vectors and compare
for name,D in [("A",docA),("B",docB)]:
    v=D.mean(axis=0); v/=np.linalg.norm(v); qv=q.mean(axis=0); qv/=np.linalg.norm(qv)
    print(f"doc {name}: single-vector cosine={qv@v:.3f}")

doc A: similarity matrix=
[[0.8 0.6 1. ]
 [0.6 0.8 0. ]]
  best per query token=[1.  0.8]  MaxSim score=1.80
doc B: similarity matrix=
[[0.  0.6]
 [1.  0.8]]
  best per query token=[0.6 1. ]  MaxSim score=1.60
doc A: single-vector cosine=0.967
doc B: single-vector cosine=0.894


In [ ]:
import numpy as np
rng=np.random.default_rng(0)
N,d,nq=20000,512,100
def make(scale,spread):
    latent=rng.normal(size=(60,d))*scale*spread
    X=latent[rng.integers(0,60,N)]+rng.normal(size=(N,d))*scale
    X=(X/np.linalg.norm(X,axis=1,keepdims=True)).astype("float32")
    Q=X[rng.choice(N,nq,replace=False)]+0.05*rng.normal(size=(nq,d)).astype("float32")*scale
    return X,(Q/np.linalg.norm(Q,axis=1,keepdims=True)).astype("float32")
def recall(top,truth): return np.mean([len(set(top[i].tolist())&set(truth[i].tolist()))/10 for i in range(nq)])
def renorm(A): return A/np.linalg.norm(A,axis=1,keepdims=True)

print("A. Matryoshka-style truncation (leading dimensions carry most of the signal)")
X,Q=make(1/np.sqrt(1+np.arange(d)/16),2.5)
truth=np.argsort(-(Q@X.T),axis=1)[:,:10]
for cut in [512,256,128,64]:
    top=np.argsort(-(renorm(Q[:,:cut])@renorm(X[:,:cut]).T),axis=1)[:,:10]
    print(f"  first {cut:>3} dims: {cut*4:>4} bytes/vector, recall@10={recall(top,truth):.3f}")

print("B. Scalar and binary quantisation (all dimensions equally informative)")
X,Q=make(np.ones(d),1.0)
truth=np.argsort(-(Q@X.T),axis=1)[:,:10]
print(f"  float32        : {d*4:>4} bytes/vector, recall@10=1.000")
sc=np.abs(X).max(axis=0)/127                         # one scale per dimension
Xi=np.round(X/sc).astype(np.int8)
top=np.argsort(-(Q@(Xi*sc).T),axis=1)[:,:10]
print(f"  int8           : {d:>4} bytes/vector, recall@10={recall(top,truth):.3f}")
Xb=X>0; Qb=Q>0
ham=(Qb[:,None,:]!=Xb[None,:,:]).sum(axis=2)         # Hamming distance = number of differing bits
top=np.argsort(ham,axis=1)[:,:10]
print(f"  binary         : {d//8:>4} bytes/vector, recall@10={recall(top,truth):.3f}")
cand=np.argsort(ham,axis=1)[:,:100]
top=np.array([cand[i][np.argsort(-(X[cand[i]]@Q[i]))[:10]] for i in range(nq)])
print(f"  binary + rescore top 100 with float vectors: recall@10={recall(top,truth):.3f}")

A. Matryoshka-style truncation (leading dimensions carry most of the signal)
  first 512 dims: 2048 bytes/vector, recall@10=1.000
  first 256 dims: 1024 bytes/vector, recall@10=0.819
  first 128 dims:  512 bytes/vector, recall@10=0.693
  first  64 dims:  256 bytes/vector, recall@10=0.570
B. Scalar and binary quantisation (all dimensions equally informative)
  float32        : 2048 bytes/vector, recall@10=1.000
  int8           :  512 bytes/vector, recall@10=0.988
  binary         :   64 bytes/vector, recall@10=0.288
  binary + rescore top 100 with float vectors: recall@10=0.812


## 6.9 Measuring Retrieval: Recall, Precision, MRR and nDCG

In [ ]:
import math
# retrieved lists (best first) and the set of truly relevant chunk ids for 5 queries
runs=[(["c1","c7","c3","c9","c2"],{"c1","c2"}),
      (["c5","c4","c8","c6","c0"],{"c4"}),
      (["c2","c9","c1","c3","c7"],{"c8"}),          # relevant chunk not retrieved at all
      (["c6","c3","c1","c4","c5"],{"c3","c4","c5"}),
      (["c8","c0","c2","c9","c1"],{"c8"})]
k=5
def recall_at(r,rel,k): return len(set(r[:k])&rel)/len(rel)
def precision_at(r,rel,k): return len(set(r[:k])&rel)/k
def rr(r,rel):
    for i,d in enumerate(r,1):
        if d in rel: return 1/i
    return 0.0
def ndcg(r,rel,k):
    dcg=sum(1/math.log2(i+1) for i,d in enumerate(r[:k],1) if d in rel)
    ideal=sum(1/math.log2(i+1) for i in range(1,min(len(rel),k)+1))
    return dcg/ideal
print("q  recall@5  precision@5  RR     nDCG@5  hit")
tot=[0,0,0,0,0]
for i,(r,rel) in enumerate(runs,1):
    v=[recall_at(r,rel,k),precision_at(r,rel,k),rr(r,rel),ndcg(r,rel,k),float(bool(set(r[:k])&rel))]
    tot=[a+b for a,b in zip(tot,v)]
    print(f"{i}  {v[0]:8.3f}  {v[1]:11.3f}  {v[2]:.3f}  {v[3]:.3f}   {int(v[4])}")
print("mean",*[f"{t/len(runs):.3f}" for t in tot])

q  recall@5  precision@5  RR     nDCG@5  hit
1     1.000        0.400  1.000  0.850   1
2     1.000        0.200  0.500  0.631   1
3     0.000        0.000  0.000  0.000   0
4     1.000        0.600  0.500  0.680   1
5     1.000        0.200  1.000  1.000   1
mean 0.800 0.280 0.600 0.632 0.800


In [ ]:
import numpy as np
from scipy.stats import binomtest
# how precise is a recall estimate from n queries?
for n in [20,50,200,1000]:
    p=0.80; se=np.sqrt(p*(1-p)/n)
    print(f"n={n:>4}: recall 0.80 +/- {1.96*se:.3f} (95% interval {p-1.96*se:.3f} to {p+1.96*se:.3f})")
# system B vs system A on the same 50 queries: B better on 5, worse on 2, same on 43
res=binomtest(5,5+2,0.5)           # ties are ignored by the sign test
print("sign test, 5 wins vs 2 losses: p =",round(res.pvalue,3))
res=binomtest(20,20+5,0.5)
print("sign test, 20 wins vs 5 losses: p =",round(res.pvalue,4))

n=  20: recall 0.80 +/- 0.175 (95% interval 0.625 to 0.975)
n=  50: recall 0.80 +/- 0.111 (95% interval 0.689 to 0.911)
n= 200: recall 0.80 +/- 0.055 (95% interval 0.745 to 0.855)
n=1000: recall 0.80 +/- 0.025 (95% interval 0.775 to 0.825)
sign test, 5 wins vs 2 losses: p = 0.453
sign test, 20 wins vs 5 losses: p = 0.0041


## 6.10 Capacity Planning, Sharding and Running a Vector Store

In [ ]:
import math
def hnsw_gb(N,d,bytes_per_dim=4.0,M=16,payload_bytes=200):
    vec=N*d*bytes_per_dim; links=N*2*M*4          # layer 0 keeps up to 2*M links, 4 bytes (int32) each
    meta=N*payload_bytes                           # id + metadata/payload per chunk
    return vec/1e9,links/1e9,meta/1e9
N,d=50_000_000,768
print(f"{'format':22} {'vectors':>8} {'links':>6} {'payload':>8} {'total GB':>9}")
options=[("float32",4.0),("float16",2.0),("int8",1.0),("PQ 96 bytes",96/768)]
for name,b in options:
    v,l,m=hnsw_gb(N,d,b)
    print(f"{name:22} {v:>8.1f} {l:>6.1f} {m:>8.1f} {v+l+m:>9.1f}")
total=sum(hnsw_gb(N,d,4.0))
ram_node=64*0.7                                    # a 64 GB machine, keep 30% free for OS and queries
shards=math.ceil(total/ram_node)
print(f"float32 index = {total:.1f} GB -> {shards} shards of 64 GB machines at 70% fill")
qps=2000; per_replica_qps=250      # measured: one shard replica answers 250 QPS at the target latency
replicas=math.ceil(qps/per_replica_qps)           # every query visits EVERY shard, so each shard must carry the full 2000 QPS
print(f"{qps} QPS / {per_replica_qps} per replica = {replicas} replicas of each shard -> {shards*replicas} machines in total")

format                  vectors  links  payload  total GB
float32                   153.6    6.4     10.0     170.0
float16                    76.8    6.4     10.0      93.2
int8                       38.4    6.4     10.0      54.8
PQ 96 bytes                 4.8    6.4     10.0      21.2
float32 index = 170.0 GB -> 4 shards of 64 GB machines at 70% fill
2000 QPS / 250 per replica = 8 replicas of each shard -> 32 machines in total


In [ ]:
# NOT RUN HERE: shape of a filtered vector query with the qdrant-client package
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")           # local, in-process; use a URL for a server
client.create_collection(
    "chunks",
    vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE),
)
client.upsert("chunks", points=[
    models.PointStruct(id=1, vector=vec_1, payload={"dept": "billing", "year": 2025}),
])
hits = client.query_points(
    "chunks",
    query=query_vector,
    query_filter=models.Filter(must=[
        models.FieldCondition(key="dept", match=models.MatchValue(value="billing")),
    ]),
    limit=5,
).points
for h in hits:
    print(h.id, round(h.score, 3), h.payload)

In [ ]:
# Cost and time to re-embed a corpus when you change the embedding model (prices are HYPOTHETICAL)
chunks=50_000_000; tokens_per_chunk=400
price_per_million=0.02            # dollars per million tokens, hypothetical
throughput=1000                   # chunks embedded per second by your pipeline, measured
total_tokens=chunks*tokens_per_chunk
cost=total_tokens/1e6*price_per_million
seconds=chunks/throughput
print(f"tokens to embed : {total_tokens:,}")
print(f"API cost        : ${cost:,.0f}")
print(f"time            : {seconds:,.0f} s = {seconds/3600:.1f} hours = {seconds/86400:.2f} days")
print(f"extra storage during the switch (old + new index, 768-d float32 vectors only): {2*chunks*768*4/1e9:.1f} GB")

tokens to embed : 20,000,000,000
API cost        : $400
time            : 50,000 s = 13.9 hours = 0.58 days
extra storage during the switch (old + new index, 768-d float32 vectors only): 307.2 GB


## 6.11 How to solve any vector databases and retrieval problem

In [ ]:
def hnsw_memory_gb(n, dim, bytes_per_number, m, meta_bytes):
    vectors = n * dim * bytes_per_number
    links = n * 2 * m * 4
    meta = n * meta_bytes
    return (vectors + links + meta) / 1e9
print(round(hnsw_memory_gb(5_000_000, 1536, 4, 32, 250), 2), "GB")
print(round(hnsw_memory_gb(5_000_000, 1536, 1, 32, 250), 2), "GB with int8")

33.25 GB
10.21 GB with int8


In [ ]:
def rrf(lists, weights, k=60):
    score = {}
    for lst, w in zip(lists, weights):
        for pos, doc in enumerate(lst, 1):
            score[doc] = score.get(doc, 0) + w / (k + pos)
    return sorted(score.items(), key=lambda x: -x[1])
bm25 = ["a", "b", "c", "d"]
vec  = ["c", "a", "e", "b"]
print("equal weights  :", [(d, round(s, 5)) for d, s in rrf([bm25, vec], [1, 1])])
print("BM25 weight 2  :", [(d, round(s, 5)) for d, s in rrf([bm25, vec], [2, 1])])

equal weights  : [('a', 0.03252), ('c', 0.03227), ('b', 0.03175), ('e', 0.01587), ('d', 0.01562)]
BM25 weight 2  : [('a', 0.04892), ('c', 0.04814), ('b', 0.04788), ('d', 0.03125), ('e', 0.01587)]


In [ ]:
import numpy as np
def doctor(X,Q=None,expected_dim=None,sample=200,seed=0):
    """Cheap sanity checks to run BEFORE blaming the index or the model."""
    rng=np.random.default_rng(seed); report=[]
    if expected_dim and X.shape[1]!=expected_dim: report.append(f"FAIL dimension {X.shape[1]} != expected {expected_dim}")
    if not np.isfinite(X).all(): report.append("FAIL NaN or inf in vectors")
    norms=np.linalg.norm(X,axis=1)
    if norms.min()==0: report.append(f"FAIL {(norms==0).sum()} zero vectors (empty text?)")
    if norms.max()-norms.min()>1e-3: report.append(f"WARN norms range {norms.min():.2f}..{norms.max():.2f}: normalise if you use dot product as cosine")
    ids=rng.choice(len(X),min(sample,len(X)),replace=False)
    S=X[ids]@X.T; self_rank1=np.mean(S.argmax(axis=1)==ids)
    if self_rank1<0.99: report.append(f"FAIL self-retrieval: only {self_rank1:.0%} of stored vectors find themselves first")
    uniq=len({row.tobytes() for row in X})
    if uniq<len(X): report.append(f"WARN {len(X)-uniq} exact duplicate vectors")
    if Q is not None and Q.shape[1]!=X.shape[1]: report.append("FAIL query dim differs from index dim")
    return report or ["all checks passed"]
rng=np.random.default_rng(1)
good=rng.normal(size=(500,32)); good/=np.linalg.norm(good,axis=1,keepdims=True)
print("healthy index:",doctor(good,expected_dim=32))
bad=rng.normal(size=(500,32))*rng.uniform(0.5,4,size=(500,1))      # not normalised
bad[10]=0; bad[20]=bad[21]                                           # an empty-text vector and a duplicate
bad[30,3]=np.nan
print("broken index:")
for line in doctor(np.nan_to_num(bad),expected_dim=32): print("  ",line)

# the fix: drop empty vectors, normalise, remove duplicates, then run the checks again
fixed=np.nan_to_num(bad)
fixed=fixed[np.linalg.norm(fixed,axis=1)>0]
fixed=fixed/np.linalg.norm(fixed,axis=1,keepdims=True)
fixed=np.unique(fixed,axis=0)
print("after the fix:",doctor(fixed,expected_dim=32))

healthy index: ['all checks passed']
broken index:
   FAIL 1 zero vectors (empty text?)
   WARN norms range 0.00..27.38: normalise if you use dot product as cosine
   FAIL self-retrieval: only 68% of stored vectors find themselves first
   WARN 1 exact duplicate vectors
after the fix: ['all checks passed']


In [ ]:
from scipy.stats import binomtest
# rank of the right chunk for 8 queries (0 = not retrieved in the top 10)
a = [1, 2, 1, 5, 1, 3, 1, 0]
b = [1, 1, 1, 2, 1, 2, 2, 4]
rr = lambda ranks: [0 if r == 0 else 1 / r for r in ranks]
mrr_a = sum(rr(a)) / len(a); mrr_b = sum(rr(b)) / len(b)
wins = sum(rb > ra for ra, rb in zip(rr(a), rr(b)))
losses = sum(rb < ra for ra, rb in zip(rr(a), rr(b)))
print(f"MRR A={mrr_a:.3f}  B={mrr_b:.3f}  difference={mrr_b-mrr_a:+.3f}")
print(f"B better on {wins} queries, worse on {losses}, tied on {len(a)-wins-losses}")
print("sign test p =", round(binomtest(wins, wins + losses, 0.5).pvalue, 3))

MRR A=0.629  B=0.719  difference=+0.090
B better on 4 queries, worse on 1, tied on 3
sign test p = 0.375


## 6.12 Going Deeper: Internals, Edge Cases, Performance and What Interviewers Probe

This part is for when you must explain not only how to use an index but why it behaves as it does. Skip it on a first reading; return before an interview or a design review.

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
M=16; ml=1/np.log(M)                               # level multiplier used by HNSW
N=1_000_000
level=np.floor(-np.log(1-rng.random(N))*ml).astype(int)    # level = floor(-ln(U) * ml)
for l in range(5):
    print(f"layer >= {l}: {np.mean(level>=l):.6f} of nodes (theory M^-{l} = {M**-l:.6f})  = {np.sum(level>=l):,} nodes")
print("highest layer used:",level.max())

layer >= 0: 1.000000 of nodes (theory M^-0 = 1.000000)  = 1,000,000 nodes
layer >= 1: 0.062675 of nodes (theory M^-1 = 0.062500)  = 62,675 nodes
layer >= 2: 0.004005 of nodes (theory M^-2 = 0.003906)  = 4,005 nodes
layer >= 3: 0.000270 of nodes (theory M^-3 = 0.000244)  = 270 nodes
layer >= 4: 0.000015 of nodes (theory M^-4 = 0.000015)  = 15 nodes
highest layer used: 5


In [ ]:
import numpy as np
# relevance of 5 candidates to the query, and how similar candidates are to each other
rel = np.array([0.90, 0.88, 0.70, 0.65, 0.60])
sim = np.array([[1.00, 0.95, 0.20, 0.30, 0.10],
                [0.95, 1.00, 0.25, 0.10, 0.10],
                [0.20, 0.25, 1.00, 0.40, 0.10],
                [0.30, 0.10, 0.40, 1.00, 0.10],
                [0.10, 0.10, 0.10, 0.10, 1.00]])
def mmr(lam, k=3):
    chosen = []
    while len(chosen) < k:
        best, best_score = None, -9
        for i in range(len(rel)):
            if i in chosen: continue
            penalty = max(sim[i][j] for j in chosen) if chosen else 0.0
            score = lam * rel[i] - (1 - lam) * penalty
            if score > best_score: best, best_score = i, score
        chosen.append(best)
        print(f"  lambda={lam}: pick d{best+1} (score {best_score:.3f})")
    return [f"d{i+1}" for i in chosen]
print("plain top 3 by relevance:", [f"d{i+1}" for i in np.argsort(-rel)[:3]])
print("MMR result:", mmr(0.7))

plain top 3 by relevance: ['d1', 'd2', 'd3']
  lambda=0.7: pick d1 (score 0.630)
  lambda=0.7: pick d3 (score 0.430)
  lambda=0.7: pick d5 (score 0.390)
MMR result: ['d1', 'd3', 'd5']


## 6.13 Mixed-Practice Drill

Ten minutes, no notes first. The questions are mixed on purpose: in real work nobody tells you which section a problem belongs to. Write your answer, then compare. If you miss one, the section in brackets is where to reread.

## 6.14 Self-Test and Scoring Guide

Sixteen questions, one point each. Do them on paper, without the chapter, in about 25 minutes. Answers follow the last question.

## Chapter recap

## Practice set

Problem 6.P.1 (easy) What does ANN stand for and what does it trade? **Answer:** Approximate Nearest Neighbour; it trades a small loss of accuracy (recall) for large speed gains.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch06.html